# Web Scraping Audi A3 (Alemania)

**Rol de canalización:** Cuaderno obligatorio 01, el paso de adquisición de datos.

Este cuaderno descarga listados de AutoScout24 y guarda el CSV sin formato utilizado en pasos posteriores. La lógica de scraping es intencionalmente directa y legible para la clase.

**Consume:** parámetros de búsqueda del contexto del cuaderno/configuración y respuestas HTML del sitio web.
**Produce:** un CSV sin formato con marca de tiempo en `data/raw`.
**Feeds:** Preprocesamiento del Notebook 02.

La ruta del archivo de salida y la convención de nomenclatura deben permanecer sin cambios porque los portátiles posteriores cargan el archivo más nuevo según ese patrón.

### Descripción general funcional
Entradas: parámetros de búsqueda (marca/modelo/país), respuestas HTML del sitio web.
Proceso: solicitar páginas, analizar tarjetas de listado, normalizar campos, armar una tabla.
Salidas: un DataFrame en memoria y un CSV escrito en `data/raw` con un nombre con marca de tiempo.
Motivo: la recopilación de datos sin procesar es el primer paso del proceso y proporciona la fuente para todos los análisis posteriores.

**Objetivo:** Explicar el objetivo integral de recopilar listados sin procesar y guardar un CSV.
**Entradas:** Parámetros de búsqueda (marca/modelo/país) y páginas HTML del sitio web.
**Proceso:** Solicitar páginas, analizar tarjetas de listado, ensamblar filas y guardar una tabla.
**Salidas:** Un CSV sin procesar en `data/raw` utilizado como punto de partida para la limpieza.
**Por qué:** Una instantánea sin formato consistente hace que el resto del proceso sea reproducible y enseñable.


## 1. Importaciones y configuración

Mantenemos la configuración en la parte superior para que los principiantes puedan cambiar la marca, el modelo o el país sin tener que buscar en el cuaderno.

### ¿Qué pasa aquí?
Cargamos solo las bibliotecas que realmente usaremos. Luego definimos los parámetros de búsqueda (marca, modelo, país) y la plantilla de URL. Mantener estos valores juntos facilita que los principiantes cambien la consulta sin tocar pasos posteriores.

### Por qué es importante esta sección
Mantenemos toda la configuración en la parte superior para que los principiantes puedan cambiar las entradas de forma segura sin modificar la lógica de análisis. Esta separación reduce los errores y aclara qué valores controlan la recopilación de datos.

**Objetivo:** Centralizar toda la configuración necesaria para el scraping.
**Insumos:** Marca, modelo, código/nombre de país y rango de páginas.
**Proceso:** Defina estos valores una vez y cree la URL base y los encabezados.
**Salidas:** Variables reutilizables a las que hacen referencia celdas posteriores.
**Por qué:** Mantener las entradas en un solo lugar reduce los errores y facilita los experimentos.


In [ ]:
# Objetivo: importar bibliotecas y definir la configuración de scraping en un solo lugar.  #Objetivo
# Entrada: nombres de biblioteca (solicitudes, BeautifulSoup, re, pandas, hora) necesarios para HTTP, análisis y guardado.  #Aporte
# Entrada: valores project_config.yaml compartidos para el alcance y las rutas de datos.  #Aporte
# Proceso: importar bibliotecas, cargar configuraciones, establecer variables de alcance, crear URL base y preparar encabezados.  #Proceso
# Salida: variables reutilizables que controlan las solicitudes y la denominación de salida sin formato.  #Producción

import requests  # Solicitudes HTTP utilizadas para descargar páginas web del sitio.
from bs4 import BeautifulSoup  # Analizador HTML utilizado para buscar y extraer etiquetas.
import re  # Expresiones regulares utilizadas para extraer números de caballos de fuerza.
import pandas as pd  # Biblioteca DataFrame utilizada para almacenar filas y guardar CSV.
import time  # Módulo de tiempo utilizado para hacer una pausa cortés entre solicitudes.
from pathlib import Path  # Path maneja rutas de archivos relativas al repositorio.


def find_repo_root(start: Path) -> Path:  # Localice la raíz del repositorio comprobando las carpetas principales.
    for p in [start] + list(start.parents):
        if (p / '.git').exists() or (p / 'config' / 'project_config.yaml').exists():
            return p
    return start


def load_project_config(config_path: Path) -> dict:  # Lector YAML ligero para clave: configuración de valor.
    config = {}
    if not config_path.exists():
        return config
    for raw_line in config_path.read_text(encoding='utf-8').splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#') or ':' not in line:
            continue
        key, value = line.split(':', 1)
        key = key.strip()
        value = value.strip()
        if value.startswith(('"', "'")) and value.endswith(('"', "'")):
            value = value[1:-1]
        elif value.lower() in ('true', 'false'):
            value = value.lower() == 'true'
        else:
            try:
                value = int(value)
            except ValueError:
                try:
                    value = float(value)
                except ValueError:
                    pass
        config[key] = value
    return config


def resolve_repo_path(repo_root: Path, config_value: str, default_path: str) -> Path:
    path_value = str(config_value or default_path)
    p = Path(path_value)
    return p if p.is_absolute() else (repo_root / p)


REPO_ROOT = find_repo_root(Path.cwd())
PROJECT_CONFIG = load_project_config(REPO_ROOT / 'config' / 'project_config.yaml')

# Configuración de búsqueda # Este bloque define lo que raspamos.
make = str(PROJECT_CONFIG.get('make', 'audi')).strip().lower()
model = str(PROJECT_CONFIG.get('model', 'a3')).strip().lower()
country_name = str(PROJECT_CONFIG.get('country', 'germany')).strip().lower()

country_code_map = {
    'germany': 'D',
    'spain': 'E',
    'france': 'F',
    'italy': 'I',
    'belgium': 'B',
    'netherlands': 'NL',
}
country_code = country_code_map.get(country_name, 'D')

start_page = int(PROJECT_CONFIG.get('start_page', 1))
end_page = int(PROJECT_CONFIG.get('end_page', 3))

RAW_DATA_DIR = resolve_repo_path(
    REPO_ROOT,
    PROJECT_CONFIG.get('raw_data_path', 'data/raw'),
    'data/raw',
)

# Plantilla de URL y encabezados # Estos son necesarios para crear una solicitud HTTP válida.
base_url = f'https://www.autoscout24.com/lst/{make}/{model}?atype=C&cy={country_code}'
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36',
    'Accept-Language': 'en-US,en;q=0.9',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
    'Referer': 'https://www.autoscout24.com/',
}


## 2. Inicie una sesión y busque una página

Primero buscamos una página para confirmar que el sitio responde y establecer una sesión (cookies, encabezados).

Esta sección es una verificación mínima de conectividad. Utiliza los mismos encabezados y estructura de URL que el scrape completo para que las fallas se detecten tempranamente.

### Por qué existe este paso
El raspado puede fallar si el sitio web bloquea requests o cambia su comportamiento. Primero hacemos una solicitud para poder detenernos temprano si no se puede acceder a la página. Esto evita ejecutar un bucle largo con datos rotos.

### Por qué es necesario este paso
El web scraping puede fallar silenciosamente. Una única solicitud verifica tempranamente la conectividad y la estructura HTML antes de recorrer muchas páginas, evitando pérdidas de tiempo y resultados incorrectos.

**Objetivo:** Confirmar que el sitio web sea accesible y que el HTML sea legible.
**Entradas:** URL base, encabezados y un objeto de sesión.
**Proceso:** Enviar una solicitud, verificar el código de estado, analizar HTML.
**Salidas:** Un objeto BeautifulSoup analizado para una página en buen estado.
**Por qué:** Fallar temprano evita bucles largos sobre respuestas rotas o bloqueadas.


In [ ]:
# Objetivo: realizar una única solicitud de prueba y analizar el HTML para confirmar el acceso.  #Objetivo
# Entrada: base_url y encabezados para crear una solicitud válida.  #Aporte
# Entrada: request.Sesión para gestionar cookies y reutilizar la configuración de conexión.  #Aporte
# Proceso: cree una URL de prueba, envíe la solicitud, valide el código de estado, analice HTML.  #Proceso
# Salida: Un objeto BeautifulSoup (sopa) que confirma que la página es legible.  #Producción

example_url = base_url + '&desc=0&page=1'  # Cree una URL concreta para la primera página.

session = requests.Session()  # Cree una sesión para reutilizar cookies y encabezados.
response = session.get(example_url, headers=headers, timeout=10)  # Descargue el HTML con un tiempo de espera.

status_code = response.status_code  # Almacene el código de estado HTTP de la respuesta.
if status_code != 200:  # Compruebe si el servidor devolvió una respuesta exitosa.
    raise ValueError(f'Request failed with status code {status_code}')  # Deténgase si la solicitud falló.

# Analizar una vez para garantizar que se pueda acceder al HTML # Esto crea un árbol HTML con capacidad de búsqueda.
soup = BeautifulSoup(response.text, 'html.parser')  # Analice el HTML de respuesta en BeautifulSoup.


## 3. Raspe listados en todas las páginas

Analizamos cada tarjeta de listado leyendo sus atributos de datos HTML. Esto es más sólido para principiantes que el análisis de texto complejo.

### Cómo funciona el raspado
Definimos dos pequeñas funciones auxiliares para leer los campos de cada tarjeta de listado. Luego recorremos cada página, analizamos el HTML y recopilamos filas en una lista. La lista se convierte en un DataFrame para que pueda guardarse como una tabla limpia.

### Qué produce este paso
El bucle crea una lista de diccionarios donde cada diccionario es una lista. La conversión de esa lista a un DataFrame produce una tabla rectangular limpia lista para exportar.

**Objetivo:** Extraer los campos de listado de cada página en una tabla estructurada.
**Entradas:** Rango de páginas de inicio/final, respuestas HTML y analizadores auxiliares.
**Proceso:** Recorrer páginas, analizar tarjetas, extraer atributos, agregar filas.
**Salidas:** `results_df`, un DataFrame con una fila por listado.
**Por qué:** Se requiere un formato de tabla limpio para guardar y procesar posteriormente.


## 3A. Asistentes de análisis (funciones de responsabilidad única)
Aislamos el análisis de tarjetas en pequeñas funciones auxiliares para mantener legible el bucle principal.


In [ ]:
# Objetivo: extraer datos de listado de cada página de resultados en un DataFrame.  #Objetivo
# Entrada: página_inicial/página_final define qué páginas se solicitan.  #Aporte
# Entrada: base_url, encabezados, sesión definen cómo se crean solicitudes.  #Aporte
# Entrada: las respuestas HTML contienen tarjetas de listado con atributos de datos.  #Aporte
# Proceso: recorrer páginas, analizar HTML, extraer campos, agregar filas y crear un DataFrame.  #Proceso
# Salida: results_df que contiene una fila por listado con columnas consistentes.  #Producción

# Ayudante para extraer caballos de fuerza del texto de la tarjeta de listado.  # Esto sigue analizando en un solo lugar.
def extract_power_hp(card):  # Defina una función para analizar los caballos de fuerza de una tarjeta.
    for span in card.select('span'):  # Busque texto de poder en todos los elementos abarcados.
        text = span.get_text(' ', strip=True)  # Normalice el texto para una coincidencia confiable.
        match = re.search(r'\((\d+)\s*hp\)', text)  # Busque un número seguido de "hp".
        if match:  # Si se encuentra un valor de caballos de fuerza,
            return match.group(1)  # Devuelve el número como una cadena.
    return None  # Si no se encuentra ningún texto sobre caballos de fuerza, devuelva Ninguno.

# Ayudante para extraer campos estructurados de una tarjeta de listado.  # Utiliza atributos de datos-*.
def parse_listing_card(card):  # Defina una función para leer atributos de datos.
    return {  # Devuelve un diccionario que representa un listado.
        'make': card.get('data-make'),  # Nombre de marca almacenado en data-make.
        'model': card.get('data-model'),  # Nombre del modelo almacenado en el modelo de datos.
        'mileage': card.get('data-mileage'),  # Valor del kilometraje almacenado en datos-kilometraje.
        'price': card.get('data-price'),  # Valor del precio almacenado en data-price.
        'price_label': card.get('data-price-label'),  # Etiqueta de precio almacenada en etiqueta-precio-datos.
        'registration': card.get('data-first-registration'),  # Texto de la fecha de registro.
        'fuel': card.get('data-fuel-type'),  # Código de tipo de combustible.
        'country': card.get('data-listing-country'),  # Código de país del listado.
        'power_hp': extract_power_hp(card),  # Potencia en hp extraída del texto.
    }  # Fin del diccionario de listado.



##3B. Paginación y extracción a nivel de página.
Recorremos las páginas de resultados, buscamos HTML, buscamos tarjetas de listado y agregamos filas analizadas.


In [ ]:
results = []  # Cree una lista para recopilar todas las filas del listado.
for page in range(start_page, end_page + 1):  # Recorra cada número de página.
    url = base_url + f'&desc=0&page={page}'  # Cree la URL completa de la página.
    response = session.get(url, headers=headers, timeout=15)  # Solicite la página HTML.
    if response.status_code != 200:  # Si la solicitud falla,
        print(f'Error {response.status_code} on {url}')  # Registre el error para solucionar problemas.
        continue  # Saltar a la página siguiente.

    soup = BeautifulSoup(response.text, 'html.parser')  # Analiza la respuesta HTML.

    # Reserva del selector porque AutoScout24 cambia el marcado con el tiempo.
    cards = []
    used_selector = None
    for sel in [
        'article[data-testid="list-item"]',
        'article.cldt-summary-full-item',
        'article[data-testid="decluttered-list-item"]',
    ]:
        candidate = soup.select(sel)
        if candidate:
            cards = candidate
            used_selector = sel
            break

    if not cards:  # Si no se encuentran tarjetas en la página,
        title = soup.title.get_text(strip=True) if soup.title else 'no-title'
        possible_block = 'captcha' in response.text.lower()
        print(f'No listings found on {url} | title={title} | captcha_hint={possible_block}')
        continue  # Saltar a la página siguiente.

    print(f'Page {page}: {len(cards)} listings using {used_selector}')

    for card in cards:  # Pase sobre cada tarjeta de listado.
        row = parse_listing_card(card)  # Extraiga campos en un diccionario.
        row['make'] = make  # Agregue la marca desde el contexto de búsqueda.
        row['model'] = model  # Agregue el modelo desde el contexto de búsqueda.
        row['country_name'] = country_name  # Agregue el contexto del nombre del país.
        row['page'] = page  # Agregue el número de página para la trazabilidad.
        results.append(row)  # Agregue la fila a la lista de resultados.

    time.sleep(1)  # Haga una pausa para reducir la carga en el servidor.


## 3C. Reunir conjunto de datos
Convertimos las filas acumuladas en un DataFrame y verificamos la estructura.


In [ ]:
results_df = pd.DataFrame(results)  # Convierta la lista de filas en un DataFrame.
results_df.head()  # Muestre las primeras filas para confirmar la estructura.


## 4. Guardar datos sin procesar

Guardamos exactamente un CSV sin formato en la carpeta de datos del repositorio. Los cuadernos posteriores dependen de este patrón de nombre de archivo.

### Salida
Guardamos una marca de tiempo CSV en la carpeta data/raw en la raíz del proyecto. Este patrón de nombre de archivo es importante porque el cuaderno de preprocesamiento carga el archivo más nuevo que coincide con él.

### Detalles de salida
El archivo de salida es CSV con una marca de tiempo en el nombre del archivo. Esto hace que cada ejecución sea rastreable y permite que los cuadernos posteriores seleccionen el último archivo por patrón.

**Objetivo:** Guardar los datos extraídos en el disco con un nombre de archivo con marca de tiempo.
**Entradas:** `results_df` y valores de configuración utilizados en el nombre del archivo.
**Proceso:** Busque la raíz del repositorio, cree la carpeta de salida, escriba CSV.
**Salidas:** Un archivo CSV en `data/raw`.
**Por qué:** El cuaderno de preprocesamiento carga el archivo sin formato más reciente según este patrón.


In [ ]:
# Objetivo: guardar el DataFrame extraído como un archivo CSV sin formato.  #Objetivo
# Entrada: results_df que contiene los listados eliminados.  #Aporte
# Entrada: marca/modelo/nombre_país y ruta sin procesar configurada para el nombre de salida.  #Aporte
# Proceso: cree una carpeta de salida, cree un nombre de archivo con marca de tiempo, guarde CSV.  #Proceso
# Salida: Un archivo CSV en data/raw (o raw_data_path configurado).  #Producción

from datetime import datetime  # Importe fecha y hora para crear una cadena de marca de tiempo.

RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)  # Cree el directorio si falta.

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
output_path = RAW_DATA_DIR / f'autoscout24_listings_{make}_{model}_{country_name}_{timestamp}.csv'

results_df.to_csv(output_path, index=False)
print('Saved to', output_path)
